In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# MLP NumPy: forward, backward, update

Bài toán XOR buộc hidden layer học biểu diễn phi tuyến.

In [ ]:
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([[0.],[1.],[1.],[0.]])
rng=np.random.default_rng(42)
W1=rng.normal(0,.5,(2,4)); b1=np.zeros((1,4)); W2=rng.normal(0,.5,(4,1)); b2=np.zeros((1,1))
sigmoid=lambda z: 1/(1+np.exp(-np.clip(z,-30,30)))

def forward(X):
    z1=X@W1+b1; h=np.tanh(z1); p=sigmoid(h@W2+b2); return h,p

lr=.5; steps=800 if FAST_MODE else 4000
for step in range(steps):
    h,p=forward(X); n=len(X)
    dz2=(p-y)/n; dW2=h.T@dz2; db2=dz2.sum(0,keepdims=True)
    dz1=(dz2@W2.T)*(1-h*h); dW1=X.T@dz1; db1=dz1.sum(0,keepdims=True)
    W1-=lr*dW1; b1-=lr*db1; W2-=lr*dW2; b2-=lr*db2
loss=-np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9))
pred=(forward(X)[1]>=.5).astype(int)
assert (pred==y).mean()==1.0
print("loss",round(loss,5),"pred",pred.ravel().tolist())

In [ ]:
# Gradient check một phần tử W2
h,p=forward(X); analytic=(h.T@((p-y)/len(X)))[0,0]
eps=1e-5; old=W2[0,0]
def loss_now():
    q=forward(X)[1]; return -np.mean(y*np.log(q+1e-9)+(1-y)*np.log(1-q+1e-9))
W2[0,0]=old+eps; plus=loss_now(); W2[0,0]=old-eps; minus=loss_now(); W2[0,0]=old
numeric=(plus-minus)/(2*eps)
assert abs(analytic-numeric)<1e-4
print("gradient check",analytic,numeric)